# Lexical shortcut audit: multi-class depression subtype classification

How much classical-ML performance on the Nusrat et al. (2024) tweet dataset
survives when self-disclosure vocabulary is removed?

**Dataset:** Zenodo 10.5281/zenodo.14233292 (CC BY 4.0), cite Nusrat, Shahzad & Jamal (2024).

**How to run**

- *Colab:* put `train.csv` in `MyDrive/Major_Project/data/`, then Runtime -> Run all.
- *Local:* run from the repo (`train.csv` is picked up from `data/` or `raw/`).

Set `SMOKE_N = 2000` in the config cell to rehearse the whole notebook on a
stratified subsample in a few minutes before committing to the full run.

**Design.** Every grid cell is fitted once, and its *out-of-fold predictions
and decision scores* are checkpointed. Every table and figure below is derived
from those stored arrays, so no metric is ever computed by refitting, and all
tables are guaranteed to be mutually consistent. Interrupted runs resume from
the checkpoints.

No deep learning. scikit-learn, NLTK and TextBlob only.

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'textblob'],
                   check=False)

import nltk
for pkg in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab', 'vader_lexicon']:
    nltk.download(pkg, quiet=True)

print('environment:', 'Colab' if IN_COLAB else 'local')

In [ ]:
import os, re, json, time, warnings
import numpy as np, pandas as pd

# Only the harmless sklearn chatter is silenced. ConvergenceWarning stays
# visible on purpose: a model that did not converge invalidates its own row.
warnings.filterwarnings('ignore', message='.*stop_words.*')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/Major_Project'
else:
    here = os.getcwd()
    ROOT = os.path.dirname(here) if os.path.basename(here) == 'notebooks' else here

RES  = f'{ROOT}/results'
CKPT = f'{RES}/checkpoints'
FIGS = f'{RES}/figures'
for d in (RES, CKPT, FIGS):
    os.makedirs(d, exist_ok=True)

CANDIDATES = [f'{ROOT}/data/train.csv', f'{ROOT}/raw/train.csv', 'train.csv']
TRAIN_CSV = next((p for p in CANDIDATES if os.path.exists(p)), None)
if TRAIN_CSV is None:
    raise FileNotFoundError('train.csv not found. Looked in:\n  ' +
                            '\n  '.join(CANDIDATES))

SEED     = 42
N_FOLDS  = 5
MAX_FEAT = 10000
TIERS    = ['M0', 'M1', 'M2', 'M3']
TASKS    = ['binary', 'subtype']
MASK     = 'xxmaskxx'

# Section 19: the C grid searched by nested CV for the two linear models, and
# how many CV repeats they get in total (repeat 0 is the main grid). Changing
# TUNE_C means deleting the *_tuned checkpoints, whose names do not record it.
TUNE_C   = [0.1, 1, 10, 100]
REPEATS  = 3

# None = full corpus. An int runs the whole notebook on a stratified
# subsample of that size, for rehearsing changes without a multi-hour wait.
SMOKE_N  = None

np.random.seed(SEED)
print('root     ', ROOT)
print('train.csv', TRAIN_CSV)
print('results  ', RES)
print('smoke    ', SMOKE_N or 'off (full corpus)')

## 1. Load and clean

Every count is printed so the paper reports the real n, not the raw file size.

One fix over the earlier version: deduplication runs on a **normalised** key
(lowercased, handles and URLs stripped, punctuation collapsed) rather than on
the raw string. Boilerplate disclosures recur with trivial punctuation
differences (`i have bipolar disorder` appears 15 times), and exact-match
deduplication leaves those copies straddling the train/test split, which
inflates every score by exactly the mechanism this study is auditing.

In [ ]:
raw = pd.read_csv(TRAIN_CSV)
print('rows in file             ', len(raw))

df = raw.dropna(subset=['Tweets', 'Labels']).copy()
print('after dropping nulls     ', len(df))

df['Tweets'] = df['Tweets'].astype(str).str.strip()
df = df[df['Tweets'].str.len() > 0]
print('after dropping empties   ', len(df))

def dedup_key(s):
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'@\w+', ' ', s)
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

key = df['Tweets'].map(dedup_key)
n_conflict = int((df.assign(k=key).groupby('k')['Labels'].nunique() > 1).sum())
print('keys carrying >1 label   ', n_conflict)

before = len(df)
df = df[~key.duplicated()].reset_index(drop=True)
print('near-duplicates removed  ', before - len(df))
print('final n                  ', len(df))

if SMOKE_N:
    df = (df.groupby('Labels', group_keys=False)
            .sample(frac=min(1.0, SMOKE_N / len(df)), random_state=SEED)
            .reset_index(drop=True))
    print(f'SMOKE_N active: subsampled to {len(df)} rows')

df['y_subtype'] = df['Labels']
df['y_binary']  = np.where(df['Labels'].str.lower().eq('no'), 'control', 'depression')

print()
print(df['Labels'].value_counts().to_string())
print()
print(df['y_binary'].value_counts().to_string())

## 2. Measured leakage (T1)

How often does a tweet contain the vocabulary its own label was retrieved by?
This is the evidence that motivates the whole study.

In [ ]:
CLASS_TERMS = {
 'postpartum'      : ['postpartum', 'post partum', 'post-partum', 'ppd'],
 'major depressive': ['major depression', 'major depressive', 'mdd'],
 'bipolar'         : ['bipolar', 'manic', 'mania'],
 'psychotic'       : ['psychotic', 'psychosis', 'hallucinat', 'delusion'],
 'atypical'        : ['atypical'],
}

low = df['Tweets'].str.lower()
rows = []
for lab in sorted(df['Labels'].unique()):
    sub = low[df['Labels'].eq(lab)]
    kws = CLASS_TERMS.get(lab, [])
    rows.append({
        'class'       : lab,
        'n'           : len(sub),
        'pct_own_term': round(100 * sub.apply(lambda s: any(k in s for k in kws)).mean(), 1)
                        if kws else np.nan,
        'pct_depress' : round(100 * sub.str.contains('depress').mean(), 1),
        'mean_chars'  : round(sub.str.len().mean(), 1),
    })
T1 = pd.DataFrame(rows).sort_values('n', ascending=False).reset_index(drop=True)
T1.to_csv(f'{RES}/T1_leakage.csv', index=False)
print(T1.to_string(index=False))

## 3. Masking tiers

- **M0** no masking
- **M1** class terms + `depress*`
- **M2** M1 + disclosure frames + medication names
- **M3** M2 + every condition phrase in the published search queries

Masking runs on the **raw text before cleaning**, so morphological variants are
caught before lemmatisation. A phrase of k words becomes k filler tokens, so
word counts are preserved exactly. The filler is a stopword for the word
vectorizer and is stripped before the character vectorizer, so it can never
become a feature itself.

The cell ends with an assertion that no targeted term survived masking. The
earlier version asserted nothing, so a regex regression would have passed
silently and quietly weakened the paper's central claim.

In [ ]:
DISCLOSURE = [
 'diagnosed with', 'diagnosis of', 'suffering from', 'struggling with',
 'i have', 'i was diagnosed', 'my therapist', 'my psychiatrist', 'my doctor',
 'antidepressant', 'antidepressants', 'prozac', 'zoloft', 'lexapro', 'sertraline',
 'fluoxetine', 'lithium', 'quetiapine', 'olanzapine', 'risperidone', 'ssri',
]

# The search queries Nusrat et al. published (their Sect. 3.1) are a
# disclosure frame plus a condition phrase ("I have" + "bipolar mood
# disorder"). The frames are already in DISCLOSURE, so M3 adds every condition
# phrase, masked wherever it occurs and not only inside a full query.
QUERY_PHRASES = [
 'major depressive disorder', 'major depression', 'major depressive episode',
 'bipolar disorder', 'bipolar depression', 'bipolar affective disorder',
 'bipolar mood disorder',
 'major depression with atypical features', 'atypical major depression',
 'hypersomnia', 'feeling sad or hopeless', 'increased appetite', 'weight gain',
 'feeling worthless',
 'psychotic depression', 'delusional depression', 'psychotic depressive disorder',
 'melancholic depression', 'psychosis',
 'postbirth depression', 'post-childbirth depression', 'maternal depression',
 'postpartum depression',
]

M1_TERMS = sorted({t for v in CLASS_TERMS.values() for t in v} | {'depress'})
M2_TERMS = sorted(set(M1_TERMS) | set(DISCLOSURE))
M3_TERMS = sorted(set(M2_TERMS) | set(QUERY_PHRASES))
TIER_TERMS = {'M1': M1_TERMS, 'M2': M2_TERMS, 'M3': M3_TERMS}

def _pattern(terms):
    # longest first, so multi-word phrases match before their own prefixes
    parts = [re.escape(t) for t in sorted(terms, key=len, reverse=True)]
    return re.compile(r'(?<![a-z])(' + '|'.join(parts) + r')[a-z]*', re.I)

PATTERNS = {tier: _pattern(t) for tier, t in TIER_TERMS.items()}

def apply_mask(text, tier):
    if tier == 'M0':
        return text
    return PATTERNS[tier].sub(
        lambda m: ' '.join([MASK] * len(m.group(0).split())), text)

for tier in TIERS:
    df[f'text_{tier}'] = df['Tweets'].map(lambda t, _t=tier: apply_mask(t, _t))

# show a row where masking actually bites, not just the first row of a class
changed = df[df['text_M0'].ne(df['text_M2'])]
ex = changed[changed['Labels'].eq('major depressive')].iloc[0]
for tier in TIERS:
    print(f'[{tier}] {ex[f"text_{tier}"][:150]}')
print()

for tier in TIERS[1:]:
    hit = (df[f'text_{tier}'].str.count(MASK) > 0).mean()
    print(f'{tier}: {100*hit:.1f}% of tweets had at least one term masked')

    # verification the earlier notebook lacked: strip the filler back out and
    # confirm nothing the tier targets is still present
    stripped = df[f'text_{tier}'].str.replace(MASK, ' ', regex=False)
    leaked = int(stripped.str.contains(PATTERNS[tier]).sum())
    assert leaked == 0, f'{tier}: {leaked} tweets still contain a targeted term'
print('\nmasking verified: no targeted term survives in any masked tier')

## 4. Cleaning and dense features

Cleaning is identical across tiers, so masking is the only manipulated
variable. Dense features are deterministic per-row transforms with no fitted
state, so they are precomputed here; only the *scaler* is fit per fold.

Two feature sets are built once and selected per grid cell:

- **v1** word TF-IDF (1-2 grams) + 13 dense features. Matches prior work.
- **v2** v1 + char_wb 3-5 grams + VADER + clinical style markers.

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob

LEM   = WordNetLemmatizer()
STOP  = set(stopwords.words('english'))
VADER = SentimentIntensityAnalyzer()

def clean(text):
    t = str(text).lower()
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)
    t = re.sub(r'@\w+', ' ', t)
    t = re.sub(r'<.*?>', ' ', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return ' '.join(LEM.lemmatize(w) for w in t.split()
                    if w not in STOP and len(w) > 1)

PHQ9 = {
 'anhedonia'  : ['interest', 'pleasure', 'bored', 'apathy', 'enjoy', 'hobby', 'meaningless'],
 'mood'       : ['depressed', 'down', 'hopeless', 'sad', 'crying', 'misery', 'empty', 'despair'],
 'sleep'      : ['insomnia', 'sleep', 'awake', 'nightmare', 'restless', 'oversleeping'],
 'fatigue'    : ['fatigue', 'tired', 'energy', 'exhausted', 'drained', 'weary', 'lethargic'],
 'appetite'   : ['appetite', 'eat', 'food', 'overeat', 'starving', 'weight', 'hungry'],
 'self_eval'  : ['failure', 'guilt', 'blame', 'ashamed', 'useless', 'worthless', 'burden'],
 'cognition'  : ['concentrating', 'focus', 'distracted', 'forgetful', 'indecisive'],
 'psychomotor': ['slow', 'sluggish', 'fidgety', 'agitated', 'pacing'],
 'suicidality': ['suicide', 'die', 'dead', 'ending', 'harm', 'kill'],
}

# First-person singular: among the most replicated depression markers
# (Rude et al. 2004; Park et al. 2012).
FP_SINGULAR = {'i', 'me', 'my', 'myself', 'mine', "i'm", "i've", "i'll", "i'd"}
# Absolutist words: Al-Mosaiwi & Johnstone (2018) report these outperform
# negative-emotion words as a marker of depression.
ABSOLUTIST = {'absolutely', 'all', 'always', 'complete', 'completely', 'constant',
              'constantly', 'definitely', 'entire', 'ever', 'every', 'everyone',
              'everything', 'full', 'must', 'never', 'nothing', 'totally', 'whole'}
NEGATION = {'no', 'not', 'never', 'none', 'nobody', 'nothing', 'neither', 'nor',
            'cannot', "can't", "won't", "don't", "didn't", "isn't", "wasn't"}

def dense_features(texts_raw, texts_clean):
    out = pd.DataFrame(index=range(len(texts_raw)))
    out['n_chars'] = [len(t) for t in texts_raw]
    out['n_words'] = [len(t.split()) for t in texts_raw]

    blob = [TextBlob(t).sentiment for t in texts_raw]
    out['polarity']     = [b.polarity for b in blob]
    out['subjectivity'] = [b.subjectivity for b in blob]
    for sym, kws in PHQ9.items():
        out[f'phq_{sym}'] = [sum(1 for w in c.split() if w in kws) for c in texts_clean]

    vs = [VADER.polarity_scores(t) for t in texts_raw]
    for k in ['neg', 'neu', 'pos', 'compound']:
        out[f'vader_{k}'] = [s[k] for s in vs]

    toks = [[w.strip('.,!?') for w in t.lower().split()] for t in texts_raw]
    for col, vocab in [('fp_singular_ratio', FP_SINGULAR),
                       ('absolutist_ratio', ABSOLUTIST),
                       ('negation_ratio', NEGATION)]:
        out[col] = [sum(1 for w in tk if w in vocab) / max(len(tk), 1) for tk in toks]
    out['exclam']     = [t.count('!') for t in texts_raw]
    out['question']   = [t.count('?') for t in texts_raw]
    out['caps_ratio'] = [sum(1 for ch in t if ch.isupper()) / max(len(t), 1)
                         for t in texts_raw]
    return out

DENSE_V1 = ['n_chars', 'n_words', 'polarity', 'subjectivity'] + \
           [f'phq_{s}' for s in PHQ9]
DENSE_V2 = DENSE_V1 + ['vader_neg', 'vader_neu', 'vader_pos', 'vader_compound',
                       'fp_singular_ratio', 'absolutist_ratio', 'negation_ratio',
                       'exclam', 'question', 'caps_ratio']
DENSE_COLS = {'v1': DENSE_V1, 'v2': DENSE_V2}

FEATS = {}
for tier in TIERS:
    t0 = time.time()
    cleaned = df[f'text_{tier}'].map(clean)
    d = dense_features(df[f'text_{tier}'].tolist(), cleaned.tolist())
    d.insert(0, 'text', cleaned.values)
    # char n-grams must not see the filler token's characters, or the shortcut
    # sneaks back in: stop_words does not apply to char analyzers
    d.insert(1, 'text_nomask', [c.replace(MASK, ' ').strip() for c in cleaned.values])
    FEATS[tier] = d
    print(f'{tier}: features built in {time.time()-t0:.0f}s')

print(f'\nv1 dense: {len(DENSE_V1)} cols | v2 dense: {len(DENSE_V2)} cols')

## 5. Pipeline and model registry

`ColumnTransformer` keeps the vectorizer and the scaler **inside** the CV loop,
so nothing is fit on held-out folds. ComplementNB needs non-negative input, so
it gets a MinMax-scaled dense block rather than a corpus-wide `+= abs(min)`
shift, which would itself leak.

The model registry is a single table. `cw=True` marks the models that accept
`class_weight`, which is what makes the class-weighting effect separable from
the feature effect later on.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_curve, auc, roc_auc_score,
                             precision_recall_curve, average_precision_score,
                             matthews_corrcoef, cohen_kappa_score)

def make_pipe(model, nonneg=False, fs='v1', dense_only=False):
    cols = DENSE_COLS[fs]
    scl  = MinMaxScaler() if nonneg else StandardScaler()
    word = TfidfVectorizer(max_features=MAX_FEAT, ngram_range=(1, 2),
                           min_df=2, stop_words=[MASK])
    if fs == 'v1':
        parts = [('w', word, 'text')]
    else:
        char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                               max_features=20000, min_df=3)
        if dense_only:
            # HistGradientBoosting rejects sparse input, so both text blocks
            # are projected to a dense subspace first
            word = Pipeline([('t', word), ('s', TruncatedSVD(150, random_state=SEED))])
            char = Pipeline([('t', char), ('s', TruncatedSVD(150, random_state=SEED))])
        parts = [('w', word, 'text'), ('c', char, 'text_nomask')]
    return Pipeline([('feat', ColumnTransformer(parts + [('num', scl, cols)])),
                     ('clf', model)])

def tune_C(est):
    # nested CV (section 19): C is chosen by an inner 3-fold search on each
    # outer-training split. The vectorizer is fitted once per outer fold,
    # which section 13 shows does not move the score on this data.
    return GridSearchCV(est, {'C': TUNE_C}, scoring='f1_macro', n_jobs=-1,
                        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED))

MODELS = {
 'LogisticRegression': dict(cw=True,  nonneg=False,
    factory=lambda cw: LogisticRegression(max_iter=3000, C=1.0, class_weight=cw)),
 'LinearSVM': dict(cw=True, nonneg=False,
    factory=lambda cw: LinearSVC(C=1.0, max_iter=10000, class_weight=cw)),
 'ComplementNB': dict(cw=False, nonneg=True,
    factory=lambda cw: ComplementNB()),
 'DecisionTree': dict(cw=True, nonneg=False,
    factory=lambda cw: DecisionTreeClassifier(random_state=SEED, class_weight=cw)),
 'RandomForest': dict(cw=True, nonneg=False,
    factory=lambda cw: RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                              random_state=SEED, class_weight=cw)),
 'KNN': dict(cw=False, nonneg=False,
    factory=lambda cw: KNeighborsClassifier(n_neighbors=15, n_jobs=-1)),
 'RBF_SVM': dict(cw=True, nonneg=False,
    factory=lambda cw: SVC(C=1.0, kernel='rbf', class_weight=cw)),
 'HistGradBoost': dict(cw=False, nonneg=False, dense_only=True,
    factory=lambda cw: HistGradientBoostingClassifier(max_iter=300, random_state=SEED)),
 'LogisticRegression_tuned': dict(cw=True, nonneg=False,
    factory=lambda cw: tune_C(LogisticRegression(max_iter=3000, class_weight=cw))),
 'LinearSVM_tuned': dict(cw=True, nonneg=False,
    factory=lambda cw: tune_C(LinearSVC(max_iter=10000, class_weight=cw))),
}

MAIN_MODELS = ['LogisticRegression', 'LinearSVM', 'ComplementNB', 'DecisionTree',
               'RandomForest', 'KNN', 'RBF_SVM']   # ordered fast -> slow
CW_MODELS   = ['LogisticRegression', 'LinearSVM']  # v1 features + class weighting
V2_MODELS   = ['LogisticRegression', 'LinearSVM', 'HistGradBoost']

def keyword_rule(texts, task):
    preds = []
    for t in texts:
        s = t.lower()
        hit = [lab for lab, kws in CLASS_TERMS.items() if any(k in s for k in kws)]
        if task == 'binary':
            preds.append('depression' if (hit or 'depress' in s) else 'control')
        else:
            preds.append(hit[0] if hit else 'no')
    return np.array(preds)

print(f'{len(MODELS)} models registered | main grid: {len(MAIN_MODELS)}')

## 6. The experiment

A grid cell is one `(task, train_tier, test_tier, features, weight, model)`
tuple. Training and test tiers are separate axes, which makes both protocols
fall out of the same runner instead of needing a special case:

- **P-sym** `train_tier == test_tier`. Can the task be learned without the shortcut?
- **P-dep** `train_tier = M0`, `test_tier in {M1, M2, M3}`. Does a normally trained
  model collapse when the shortcut is taken away at test time?

Each cell stores its out-of-fold `pred`, decision `score`, and `fold` index.
Storing the predictions rather than the summary statistics is what lets the
tables in sections 8-13 be derived rather than recomputed: the earlier version
refit Logistic Regression three separate times to draw confusion matrices, ROC
curves and coefficient plots, and used two different model sets for two tables
that were supposed to decompose the same number.

Models without `predict_proba` (LinearSVC, SVC) contribute `decision_function`
margins instead. ROC and PR only need a ranking, so this is sound; the stored
`score_kind` records which is which.

In [ ]:
def cfg_key(c):
    key = '__'.join([c['task'], c['train_tier'], c['test_tier'],
                     c['features'], c['weight'], c['model']])
    # extra CV repeats (section 19) get their own checkpoints; repeat 0 is the
    # main grid and keeps its original name
    return key + (f'__r{c["repeat"]}' if c.get('repeat') else '')

def decision_scores(pipe, X):
    clf = pipe.named_steps['clf']
    if hasattr(clf, 'predict_proba'):
        return pipe.predict_proba(X), 'proba'
    d = pipe.decision_function(X)
    return (np.column_stack([-d, d]) if d.ndim == 1 else d), 'margin'

def run_cell(c):
    path = f'{CKPT}/{cfg_key(c)}.npz'
    if os.path.exists(path):
        z = np.load(path)
        return dict(pred=z['pred'], score=z['score'], fold=z['fold'],
                    classes=[str(x) for x in z['classes']],
                    score_kind=str(z['score_kind']), secs=float(z['secs']),
                    best_c=z['best_c'] if 'best_c' in z.files else None,
                    cached=True)

    y       = df[f'y_{c["task"]}'].values
    classes = [str(x) for x in np.unique(y)]
    X_train = FEATS[c['train_tier']]
    X_test  = FEATS[c['test_tier']]
    spec    = MODELS[c['model']]
    cw      = 'balanced' if c['weight'] == 'balanced' else None

    pred  = np.empty(len(y), dtype='U24')
    score = np.zeros((len(y), len(classes)))
    fold  = np.zeros(len(y), dtype=int)
    best_c = np.full(N_FOLDS, np.nan)   # filled only by the tuned models
    kind  = 'proba'

    t0 = time.time()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                          random_state=SEED + c.get('repeat', 0))
    for k, (tr, te) in enumerate(skf.split(X_train, y)):
        pipe = make_pipe(spec['factory'](cw), spec['nonneg'], c['features'],
                         spec.get('dense_only', False))
        pipe.fit(X_train.iloc[tr], y[tr])
        best_c[k] = getattr(pipe.named_steps['clf'], 'best_params_', {}).get('C', np.nan)
        assert list(pipe.classes_) == classes, 'fold is missing a class'
        pred[te]  = pipe.predict(X_test.iloc[te])
        score[te], kind = decision_scores(pipe, X_test.iloc[te])
        fold[te]  = k
    secs = time.time() - t0

    np.savez(path, pred=pred, score=score, fold=fold,
             classes=np.array(classes), score_kind=kind, secs=secs, best_c=best_c)
    return dict(pred=pred, score=score, fold=fold, classes=classes,
                score_kind=kind, secs=secs, best_c=best_c, cached=False)

GRID = []
for task in TASKS:
    for tier in TIERS:
        for m in MAIN_MODELS:                       # main v1 grid, P-sym
            GRID.append(dict(task=task, train_tier=tier, test_tier=tier,
                             features='v1', weight='none', model=m))
        if tier != 'M0':                            # P-dep
            for m in MAIN_MODELS:
                GRID.append(dict(task=task, train_tier='M0', test_tier=tier,
                                 features='v1', weight='none', model=m))
        for m in CW_MODELS:                         # v1 features + class weighting
            GRID.append(dict(task=task, train_tier=tier, test_tier=tier,
                             features='v1', weight='balanced', model=m))
        for m in V2_MODELS:                         # v2 features + class weighting
            GRID.append(dict(task=task, train_tier=tier, test_tier=tier,
                             features='v2', weight='balanced', model=m))

print(f'{len(GRID)} grid cells x {N_FOLDS} folds')

In [ ]:
OOF = {}
t_start = time.time()
for i, c in enumerate(GRID, 1):
    r = run_cell(c)
    OOF[cfg_key(c)] = (c, r)
    tag = 'cached' if r['cached'] else f'{r["secs"]:.0f}s'
    acc = accuracy_score(df[f'y_{c["task"]}'].values, r['pred'])
    print(f'[{i:3d}/{len(GRID)}] {cfg_key(c):<70s} acc={acc:.4f}  {tag}')
print(f'\ngrid complete in {(time.time()-t_start)/60:.1f} min')

### Deriving every metric from the stored predictions

`ALL` is the single source of truth for the rest of the notebook. Fold-level
macro-F1 is recovered from the stored `fold` index, so error bars and paired
significance tests come for free and stay consistent with the pooled numbers.

Two corrections here: the standard deviation across folds is now the sample
standard deviation (`ddof=1`), and every table downstream reads this one frame
instead of maintaining its own model subset.

In [ ]:
def per_fold_macro_f1(y, pred, fold):
    return [f1_score(y[fold == k], pred[fold == k], average='macro', zero_division=0)
            for k in range(N_FOLDS)]

rows = []
for k, (c, r) in OOF.items():
    y = df[f'y_{c["task"]}'].values
    folds = per_fold_macro_f1(y, r['pred'], r['fold'])
    rows.append({**c, 'protocol': 'P-sym' if c['train_tier'] == c['test_tier'] else 'P-dep',
                 'macro_f1': float(np.mean(folds)),
                 'macro_f1_std': float(np.std(folds, ddof=1)),
                 'accuracy': float(accuracy_score(y, r['pred'])),
                 'folds': [float(x) for x in folds],
                 'per_class_labels': r['classes'],
                 'per_class_f1': [float(x) for x in
                                  f1_score(y, r['pred'], average=None,
                                           labels=r['classes'], zero_division=0)],
                 'secs': r['secs']})
ALL = pd.DataFrame(rows)
ALL.to_csv(f'{RES}/raw_results.csv', index=False)

def pick(features='v1', weight='none', protocol='P-sym'):
    q = ALL[ALL.features.eq(features) & ALL.weight.eq(weight) &
            ALL.protocol.eq(protocol)]
    return q.rename(columns={'test_tier': 'tier'})

R  = pick()                                   # main v1 grid
RD = pick(protocol='P-dep')                   # train unmasked, test masked
RC = pick(weight='balanced')                  # v1 + class weighting
R2 = pick(features='v2', weight='balanced')   # v2 + class weighting
R2.to_csv(f'{RES}/raw_results_v2.csv', index=False)

print('rows per condition:',
      {'v1 P-sym': len(R), 'v1 P-dep': len(RD), 'v1+cw': len(RC), 'v2+cw': len(R2)})
print('\nslowest cells:')
print(ALL.nlargest(5, 'secs')[['task', 'test_tier', 'features', 'model', 'secs']]
        .to_string(index=False))

## 7. Baselines

The keyword rule is the number that matters: it does no learning at all, so
any model that fails to beat it is contributing nothing beyond the retrieval
vocabulary.

In [ ]:
base_rows = []
for task in TASKS:
    y = df[f'y_{task}'].values
    maj = pd.Series(y).value_counts().idxmax()
    for name, pred in [('majority', np.full(len(y), maj)),
                       ('keyword_rule', keyword_rule(df['Tweets'].tolist(), task))]:
        base_rows.append({'task': task, 'baseline': name,
                          'macro_f1': f1_score(y, pred, average='macro', zero_division=0),
                          'accuracy': accuracy_score(y, pred)})
B = pd.DataFrame(base_rows)
B.to_csv(f'{RES}/T_baselines.csv', index=False)
print(B.round(4).to_string(index=False))

## 8. Figure style

Defined once, near the top, rather than halfway down the notebook. The earlier
version also set `matplotlib.use('Agg')`, which silently disabled inline
display: every figure was written to disk and closed without ever being shown
in the notebook. `save()` now writes both formats *and* displays.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110,
})

# Okabe-Ito, colourblind safe
CB = {'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73', 'red': '#D55E00',
      'purple': '#CC79A7', 'sky': '#56B4E9', 'yellow': '#F0E442', 'grey': '#999999'}
TIER_C  = {'M0': CB['blue'], 'M1': CB['sky'], 'M2': CB['orange'], 'M3': CB['purple']}
PALETTE = [CB[k] for k in ['blue', 'orange', 'green', 'red', 'purple', 'sky',
                           'yellow', 'grey']]

def save(fig, name):
    fig.savefig(f'{FIGS}/{name}.pdf', bbox_inches='tight')
    fig.savefig(f'{FIGS}/{name}.png', dpi=300, bbox_inches='tight')
    print('saved', name)
    plt.show()

print('figure style ready')

## 9. The corpus and the leakage that motivates the study (V1)

Left: class sizes. Right: how often a tweet contains the vocabulary its own
label was retrieved by. The control bar is the one to look at.

In [ ]:
t1 = T1.sort_values('n', ascending=False)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3.6))

ax1.barh(t1['class'][::-1], t1['n'][::-1], color=CB['blue'])
for i, v in enumerate(t1['n'][::-1]):
    ax1.text(v + t1['n'].max() * 0.01, i, str(int(v)), va='center', fontsize=8)
ax1.set_xlabel('tweets'); ax1.set_title('(a) Corpus composition')
ax1.set_xlim(0, t1['n'].max() * 1.18)

y = np.arange(len(t1)); h = 0.38
ax2.barh(y + h/2, t1['pct_own_term'].fillna(0).values[::-1], h,
         label='contains own label term', color=CB['orange'])
ax2.barh(y - h/2, t1['pct_depress'].values[::-1], h,
         label='contains "depress*"', color=CB['purple'])
ax2.set_yticks(y); ax2.set_yticklabels(t1['class'][::-1])
ax2.set_xlabel('% of class'); ax2.set_title('(b) Retrieval-vocabulary leakage')
ax2.legend(loc='lower right'); ax2.grid(axis='x', alpha=.3)

fig.tight_layout()
save(fig, 'V1_corpus_and_leakage')

## 10. Main results (T2) and the significance of the drop

`T2` is the headline table: macro-F1 per model per tier, with the M0 -> M2
delta. The earlier version reported the delta with no test attached, so a drop
of 0.01 and a drop of 0.24 were presented identically. Since fold-level scores
are now stored, the same five folds can be paired across tiers and the drop
tested directly. With `N_FOLDS = 5` this is a low-powered test, so it is
reported as a sanity check on the direction and magnitude, not as the claim.

In [ ]:
from scipy.stats import ttest_rel

T2 = R.pivot_table(index=['task', 'model'], columns='tier',
                   values='macro_f1').round(4)
T2['delta_M0_M2'] = (T2['M0'] - T2['M2']).round(4)
T2['pct_drop']    = (100 * (T2['M0'] - T2['M2']) / T2['M0']).round(1)
T2['delta_M0_M3'] = (T2['M0'] - T2['M3']).round(4)

sig = []
for (task, model), _ in T2.iterrows():
    f0 = R[R.task.eq(task) & R.model.eq(model) & R.tier.eq('M0')]['folds'].iloc[0]
    f2 = R[R.task.eq(task) & R.model.eq(model) & R.tier.eq('M2')]['folds'].iloc[0]
    f3 = R[R.task.eq(task) & R.model.eq(model) & R.tier.eq('M3')]['folds'].iloc[0]
    t, p = ttest_rel(f0, f2)
    t3, p3 = ttest_rel(f0, f3)
    sig.append({'task': task, 'model': model, 't': round(float(t), 2),
                'p_paired': round(float(p), 4), 't_M3': round(float(t3), 2),
                'p_paired_M3': round(float(p3), 4)})
T2 = T2.join(pd.DataFrame(sig).set_index(['task', 'model']))
T2.to_csv(f'{RES}/T2_main_results.csv')
print(T2.to_string())
print(f'\npaired t-test over {N_FOLDS} folds, M0 vs M2, per model')
print('models with a significant drop at p<.05:',
      int((T2['p_paired'] < .05).sum()), 'of', len(T2))

In [ ]:
fig, axes = plt.subplots(len(TASKS), 1, figsize=(9.5, 7.2), sharex=True)
axes = np.atleast_1d(axes)
for ax, task in zip(axes, TASKS):
    sub = R[R.task.eq(task)]
    x = np.arange(len(MAIN_MODELS)); w = 0.8 / len(TIERS)
    for i, tier in enumerate(TIERS):
        vals = [sub[sub.model.eq(m) & sub.tier.eq(tier)]['macro_f1'].iloc[0]
                for m in MAIN_MODELS]
        errs = [sub[sub.model.eq(m) & sub.tier.eq(tier)]['macro_f1_std'].iloc[0]
                for m in MAIN_MODELS]
        ax.bar(x + (i - (len(TIERS) - 1) / 2) * w, vals, w, yerr=errs, capsize=2.5,
               label=tier, color=TIER_C[tier])
    kw = B[B.task.eq(task) & B.baseline.eq('keyword_rule')]['macro_f1'].iloc[0]
    ax.axhline(kw, ls='--', color=CB['red'], lw=1.2, label=f'keyword rule ({kw:.3f})')
    ax.set_xticks(x); ax.set_xticklabels(MAIN_MODELS, rotation=20, ha='right')
    ax.set_ylabel('macro-F1'); ax.set_title(task)
    ax.grid(axis='y', alpha=.3); ax.set_ylim(0, 1.05)
axes[0].legend(ncol=len(TIERS) + 1, loc='lower right')
fig.tight_layout()
save(fig, 'V6_all_models')

## 11. Per-subtype shortcut dependence

Tests the AlSaad et al. claim that near-ceiling postpartum and psychotic
performance reflects lexically salient indicators rather than symptom signal.

The earlier version titled this figure "per-subtype macro-F1", which is a
contradiction in terms: these are per-class F1 scores, and calling them macro
invites a reader to compare them against the macro-F1 in T2.

In [ ]:
REF_MODEL = 'LogisticRegression'

sub = R[R.task.eq('subtype') & R.model.eq(REF_MODEL)]
labels = sub.iloc[0]['per_class_labels']
H = pd.DataFrame({tier: sub[sub.tier.eq(tier)].iloc[0]['per_class_f1']
                  for tier in TIERS}, index=labels)
H['drop_M0_M2'] = (H['M0'] - H['M2']).round(4)
H['drop_M0_M3'] = (H['M0'] - H['M3']).round(4)
H.round(4).to_csv(f'{RES}/F2_per_class.csv')
print(f'per-class F1, subtype task, {REF_MODEL}')
print(H.round(4).to_string())

## 12. Train unmasked, test masked (P-dep)

P-sym asks whether the task is learnable without the shortcut. P-dep asks the
sharper deployment question: take a model trained the way the literature
trains it, and remove the shortcut only at test time. This is what happens
when a model trained on self-disclosure tweets meets text where nobody
announces a diagnosis.

In [ ]:
T8 = (RD.pivot_table(index=['task', 'model'], columns='tier', values='macro_f1')
        .rename(columns=lambda t: f'P-dep_{t}'))
for tier in ['M0', 'M2', 'M3']:
    T8[f'P-sym_{tier}'] = [R[R.task.eq(t) & R.model.eq(m) & R.tier.eq(tier)]['macro_f1'].iloc[0]
                           for t, m in T8.index]
T8 = T8[['P-sym_M0', 'P-sym_M2', 'P-sym_M3', 'P-dep_M1', 'P-dep_M2', 'P-dep_M3']].round(4)
for tier in ['M2', 'M3']:
    T8[f'sym_minus_dep_{tier}'] = (T8[f'P-sym_{tier}'] - T8[f'P-dep_{tier}']).round(4)
T8.to_csv(f'{RES}/T8_protocol_comparison.csv')
print(T8.to_string())
print('\nA large sym_minus_dep_M2 means retraining recovers ground that a')
print('model trained on unmasked text cannot reach at test time.')

## 13. Cost of the leaky pipeline (T3)

Fits the vectorizer on the full corpus before splitting, which is the pattern
in much published work, and reports the inflation against the correct
ordering. Both arms use the same estimator, the same folds and the same
features, so the difference is attributable to the fit/split ordering alone.

In [ ]:
from scipy.sparse import hstack, csr_matrix

leak_rows = []
for task in TASKS:
    y = df[f'y_{task}'].values
    X = FEATS['M0']
    vec = TfidfVectorizer(max_features=MAX_FEAT, ngram_range=(1, 2), min_df=2,
                          stop_words=[MASK])
    Xall = hstack([vec.fit_transform(X['text']),
                   csr_matrix(StandardScaler().fit_transform(X[DENSE_V1]))]).tocsr()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    leaky = cross_val_score(LogisticRegression(max_iter=3000), Xall, y,
                            cv=skf, scoring='f1_macro').mean()
    correct = R[R.task.eq(task) & R.tier.eq('M0') &
                R.model.eq(REF_MODEL)]['macro_f1'].iloc[0]
    leak_rows.append({'task': task, 'correct': round(correct, 4),
                      'leaky': round(float(leaky), 4),
                      'inflation': round(float(leaky) - correct, 4)})
T3 = pd.DataFrame(leak_rows)
T3.to_csv(f'{RES}/T3_leakage_cost.csv', index=False)
print(T3.to_string(index=False))

## 14. Recovering shortcut-free signal (T4, T5)

The target is the masked tiers (**M2**, **M3**), not M0. Raising M0 would only mean exploiting the
retrieval artifact more efficiently.

The earlier version had a real accounting bug here. `T4` compared the best of
{LogReg, LinearSVM, RandomForest} under v1 against the best of three different
models under v2, while `T5` decomposed the "same" gain using the best of
{LogReg, LinearSVM}. The two tables therefore described different quantities
and did not add up, and taking a max over models folds model selection into a
number that is supposed to measure a feature change.

`T5` is now computed **per model**, over the two models present in all three
conditions, so `gain_class_weight + gain_features` reconstructs the total
gain exactly. `T4` keeps its filename and best-of columns for the paper, but
now records which model won each cell and carries a paired single-model
comparison alongside.

In [ ]:
def best_by(frame):
    idx = frame.groupby(['task', 'tier'])['macro_f1'].idxmax()
    out = frame.loc[idx].set_index(['task', 'tier'])
    return out['macro_f1'], out['model']

v1_best, v1_who = best_by(R)
v2_best, v2_who = best_by(R2)
lr = lambda frame: (frame[frame.model.eq(REF_MODEL)]
                    .set_index(['task', 'tier'])['macro_f1'])

T4 = pd.DataFrame({'v1_best': v1_best, 'v1_best_model': v1_who,
                   'v2_best': v2_best, 'v2_best_model': v2_who,
                   f'v1_{REF_MODEL}': lr(R), f'v2_{REF_MODEL}': lr(R2)})
T4['gain_best']   = (T4['v2_best'] - T4['v1_best']).round(4)
T4['gain_paired'] = (T4[f'v2_{REF_MODEL}'] - T4[f'v1_{REF_MODEL}']).round(4)
T4 = T4.round(4)
T4.to_csv(f'{RES}/T4_v1_vs_v2.csv')
print(T4.to_string())

print('\n=== THE NUMBER THAT MATTERS ===')
for task in TASKS:
    m0 = T4.loc[(task, 'M0'), 'v1_best']
    print(f'{task}:')
    print(f'  M0 v1 (inflated reference)   {m0:.4f}')
    for tier in ['M2', 'M3']:
        mt, mtv2 = T4.loc[(task, tier), 'v1_best'], T4.loc[(task, tier), 'v2_best']
        print(f'  {tier} v1 (shortcut removed)     {mt:.4f}   drop {mt-m0:+.4f}')
        print(f'  {tier} v2 (signal recovered)     {mtv2:.4f}   gain {mtv2-mt:+.4f}'
              f'   residual gap to M0 {m0-mtv2:+.4f}')

In [ ]:
frames = {'v1': R, 'v1_cw': RC, 'v2_cw': R2}
rows = []
for task in TASKS:
    for tier in TIERS:
        for m in CW_MODELS:
            vals = {k: f[f.task.eq(task) & f.tier.eq(tier) & f.model.eq(m)]['macro_f1'].iloc[0]
                    for k, f in frames.items()}
            rows.append({'task': task, 'tier': tier, 'model': m, **vals,
                         'gain_class_weight': vals['v1_cw'] - vals['v1'],
                         'gain_features': vals['v2_cw'] - vals['v1_cw'],
                         'gain_total': vals['v2_cw'] - vals['v1']})
T5full = pd.DataFrame(rows).set_index(['task', 'tier', 'model'])
resid = (T5full['gain_class_weight'] + T5full['gain_features']
         - T5full['gain_total']).abs().max()
T5 = T5full.round(4)
T5.to_csv(f'{RES}/T5_gain_decomposition.csv')
print(T5.to_string())

print(f'\ndecomposition closes exactly (max residual {resid:.2e})')
print('\n=== ATTRIBUTION AT M2 ===')
for task in TASKS:
    for m in CW_MODELS:
        r = T5.loc[(task, 'M2', m)]
        print(f'{task:8s} {m:20s} class weighting {r["gain_class_weight"]:+.4f}'
              f'  |  features {r["gain_features"]:+.4f}')

In [ ]:
fig, axes = plt.subplots(1, len(TASKS), figsize=(11, 4.2), sharey=True)
axes = np.atleast_1d(axes)
L = len(TIERS) - 1                            # bar index of the strictest tier
for ax, task in zip(axes, TASKS):
    vals = ([T4.loc[(task, t), 'v1_best'] for t in TIERS]
            + [T4.loc[(task, TIERS[-1]), 'v2_best']])
    names = (['M0\n(reported)'] + TIERS[1:-1]
             + [f'{TIERS[-1]}\n(shortcut\nremoved)', f'{TIERS[-1]} + v2\n(recovered)'])
    colors = [TIER_C[t] for t in TIERS] + [CB['green']]
    ax.bar(range(L + 2), vals, color=colors, width=.68)
    for i, v in enumerate(vals):
        ax.text(i, v + .012, f'{v:.3f}', ha='center', fontsize=8.5, fontweight='bold')
    ax.annotate('', xy=(L, vals[L]), xytext=(0, vals[0]),
                arrowprops=dict(arrowstyle='->', color=CB['red'], lw=1.6))
    ax.text(L / 2, (vals[0] + vals[L]) / 2 + .03, f'{vals[L]-vals[0]:+.3f}',
            color=CB['red'], ha='center', fontsize=8.5, fontweight='bold')
    ax.annotate('', xy=(L + 1, vals[L + 1]), xytext=(L, vals[L]),
                arrowprops=dict(arrowstyle='->', color=CB['green'], lw=1.6))
    ax.text(L + .5, (vals[L] + vals[L + 1]) / 2 + .03, f'{vals[L + 1]-vals[L]:+.3f}',
            color=CB['green'], ha='center', fontsize=8.5, fontweight='bold')
    kw = B[B.task.eq(task) & B.baseline.eq('keyword_rule')]['macro_f1'].iloc[0]
    ax.axhline(kw, ls='--', color=CB['red'], lw=1.2)
    ax.text(L + 1.45, kw + .012, f'keyword rule {kw:.3f}', ha='right', fontsize=7.5,
            color=CB['red'])
    ax.set_xticks(range(L + 2)); ax.set_xticklabels(names, fontsize=8)
    ax.set_title(task); ax.set_ylim(0, 1.08); ax.grid(axis='y', alpha=.3)
axes[0].set_ylabel('macro-F1')
fig.suptitle('Reported performance, shortcut-free performance, and recovery', y=1.03)
fig.tight_layout()
save(fig, 'V4_headline_summary')

## 15. Evaluation matrix: ROC, confusion, precision / recall / F1

Macro-F1 alone hides *how* the classifiers fail. Everything in this section is
computed from the out-of-fold predictions already stored in section 6, for the
reference model at each tier, so nothing is refit.

| Output | Contents |
|---|---|
| `T6_evaluation_matrix.csv` | accuracy, balanced accuracy, precision / recall / F1 (macro and weighted), ROC-AUC, PR-AUC, MCC, Cohen kappa |
| `T7_per_class_metrics.csv` | precision / recall / F1 / support per class per tier |
| `V7_roc_pr_binary` | ROC and precision-recall curves, every tier |
| `V8_roc_pr_subtype` | one-vs-rest ROC and PR per subtype, every tier |
| `V2_confusion_M0_vs_M2`, `V9_confusion_*` | confusion matrices |
| `V10_metric_heatmap_*` | per-class precision / recall / F1 as a class x tier matrix |

In [ ]:
POS_LABEL = 'depression'

def oof(task, tier, model=None, features='v1', weight='none'):
    c, r = OOF[cfg_key(dict(task=task, train_tier=tier, test_tier=tier,
                            features=features, weight=weight,
                            model=model or REF_MODEL))]
    return r

def _normalise(s):
    # roc_auc_score(multi_class='ovr') requires rows that sum to 1; margins do
    # not, and a softmax preserves the per-row ranking that AUC depends on
    if np.allclose(s.sum(axis=1), 1, atol=1e-6):
        return s
    e = np.exp(s - s.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def metric_row(task, tier):
    r = oof(task, tier)
    y, pred, score, classes = (df[f'y_{task}'].values, r['pred'], r['score'], r['classes'])
    p_ma, r_ma, f_ma, _ = precision_recall_fscore_support(y, pred, average='macro',
                                                          zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(y, pred, average='weighted',
                                                       zero_division=0)
    if len(classes) == 2:
        pos = classes.index(POS_LABEL)
        yb = (y == POS_LABEL).astype(int)
        roc, prauc = roc_auc_score(yb, score[:, pos]), average_precision_score(yb, score[:, pos])
    else:
        roc = roc_auc_score(y, _normalise(score), multi_class='ovr',
                            average='macro', labels=classes)
        prauc = average_precision_score(label_binarize(y, classes=classes), score,
                                        average='macro')
    return {'task': task, 'tier': tier, 'n': len(y),
            'accuracy': accuracy_score(y, pred),
            'balanced_acc': balanced_accuracy_score(y, pred),
            'precision_macro': p_ma, 'recall_macro': r_ma, 'f1_macro': f_ma,
            'precision_weighted': p_w, 'recall_weighted': r_w, 'f1_weighted': f_w,
            'roc_auc': roc, 'pr_auc': prauc,
            'mcc': matthews_corrcoef(y, pred), 'kappa': cohen_kappa_score(y, pred)}

T6 = (pd.DataFrame([metric_row(t, ti) for t in TASKS for ti in TIERS])
        .set_index(['task', 'tier']).round(4))
T6.to_csv(f'{RES}/T6_evaluation_matrix.csv')
print(T6.to_string())

for tier in ['M2', 'M3']:
    print(f'\n=== COST OF REMOVING THE SHORTCUT (M0 -> {tier}) ===')
    for task in TASKS:
        a, b = T6.loc[(task, 'M0')], T6.loc[(task, tier)]
        print(f'[{task}]')
        for m in ['accuracy', 'balanced_acc', 'precision_macro', 'recall_macro',
                  'f1_macro', 'roc_auc', 'pr_auc', 'mcc', 'kappa']:
            print(f'  {m:<18} {a[m]:.4f} -> {b[m]:.4f}   {b[m]-a[m]:+.4f}'
                  f'   ({100*(b[m]-a[m])/a[m]:+.1f}%)')
        print()

In [ ]:
rows = []
for task in TASKS:
    for tier in TIERS:
        r = oof(task, tier)
        rep = classification_report(df[f'y_{task}'].values, r['pred'],
                                    labels=r['classes'], output_dict=True,
                                    zero_division=0)
        for cls in r['classes']:
            m = rep[cls]
            rows.append({'task': task, 'tier': tier, 'class': cls,
                         'precision': round(m['precision'], 4),
                         'recall': round(m['recall'], 4),
                         'f1': round(m['f1-score'], 4),
                         'support': int(m['support'])})
T7 = pd.DataFrame(rows)
T7.to_csv(f'{RES}/T7_per_class_metrics.csv', index=False)

for task in TASKS:
    print(f'===== {task} =====')
    piv = T7[T7.task.eq(task)].pivot_table(index='class', columns='tier',
                                           values=['precision', 'recall', 'f1'])
    print(piv.reindex(columns=pd.MultiIndex.from_product(
        [['precision', 'recall', 'f1'], TIERS])).round(4).to_string())
    print()

print('===== full classification reports (M0, M2, M3) =====')
for task in TASKS:
    for tier in ['M0', 'M2', 'M3']:
        r = oof(task, tier)
        print(f'--- {task} / {tier} ---')
        print(classification_report(df[f'y_{task}'].values, r['pred'],
                                    labels=r['classes'], digits=4, zero_division=0))

In [ ]:
prev = (df['y_binary'].values == POS_LABEL).mean()
fig, (axR, axP) = plt.subplots(1, 2, figsize=(10, 4.3))
for tier in TIERS:
    r = oof('binary', tier)
    pos = r['classes'].index(POS_LABEL)
    yb, s = (df['y_binary'].values == POS_LABEL).astype(int), r['score'][:, pos]
    fpr, tpr, _ = roc_curve(yb, s)
    axR.plot(fpr, tpr, color=TIER_C[tier], lw=1.9,
             label=f'{tier}  AUC {auc(fpr, tpr):.3f}')
    pr, rc, _ = precision_recall_curve(yb, s)
    axP.plot(rc, pr, color=TIER_C[tier], lw=1.9,
             label=f'{tier}  AP {average_precision_score(yb, s):.3f}')

axR.plot([0, 1], [0, 1], ls=':', color=CB['grey'], lw=1.3, label='chance')
axR.set_xlabel('false positive rate'); axR.set_ylabel('true positive rate')
axR.set_title('(a) ROC - depression vs control')
axR.legend(loc='lower right'); axR.grid(alpha=.3)

axP.axhline(prev, ls=':', color=CB['grey'], lw=1.3, label=f'prevalence {prev:.3f}')
axP.set_xlabel('recall'); axP.set_ylabel('precision')
axP.set_title('(b) Precision-recall'); axP.set_ylim(0, 1.02)
axP.legend(loc='lower left'); axP.grid(alpha=.3)

fig.suptitle(f'Binary task, {REF_MODEL}: discrimination before and after masking', y=1.02)
fig.tight_layout()
save(fig, 'V7_roc_pr_binary')

In [ ]:
labs  = oof('subtype', 'M0')['classes']
CLS_C = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(labs)}
GRID_X = np.linspace(0, 1, 200)

fig, axes = plt.subplots(2, len(TIERS), figsize=(4.3 * len(TIERS), 8), sharex=True, sharey=True)
for j, tier in enumerate(TIERS):
    r = oof('subtype', tier)
    Yb = label_binarize(df['y_subtype'].values, classes=r['classes'])
    axR, axP = axes[0, j], axes[1, j]
    tprs = []
    for i, cls in enumerate(r['classes']):
        fpr, tpr, _ = roc_curve(Yb[:, i], r['score'][:, i])
        tprs.append(np.interp(GRID_X, fpr, tpr))
        axR.plot(fpr, tpr, color=CLS_C[cls], lw=1.4,
                 label=f'{cls} ({auc(fpr, tpr):.3f})')
        ap = average_precision_score(Yb[:, i], r['score'][:, i])
        pr, rc, _ = precision_recall_curve(Yb[:, i], r['score'][:, i])
        axP.plot(rc, pr, color=CLS_C[cls], lw=1.4, label=f'{cls} ({ap:.3f})')
    macro = np.mean(tprs, axis=0)
    axR.plot(GRID_X, macro, ls='--', color='k', lw=1.9,
             label=f'macro ({auc(GRID_X, macro):.3f})')
    axR.plot([0, 1], [0, 1], ls=':', color=CB['grey'], lw=1.1)
    axR.set_title(f'{tier} - ROC (AUC)'); axR.grid(alpha=.3)
    axR.legend(fontsize=6.5, loc='lower right')
    axP.set_title(f'{tier} - precision-recall (AP)'); axP.grid(alpha=.3)
    axP.legend(fontsize=6.5, loc='lower left')
    axP.set_xlabel('false positive rate / recall')
axes[0, 0].set_ylabel('true positive rate')
axes[1, 0].set_ylabel('precision')
fig.suptitle(f'Subtype task, {REF_MODEL}: one-vs-rest discrimination by tier', y=1.0)
fig.tight_layout()
save(fig, 'V8_roc_pr_subtype')

In [ ]:
def plot_confusion(task, tiers, name, title=None):
    labs = oof(task, tiers[0])['classes']
    k = len(labs)
    fig, axes = plt.subplots(1, len(tiers), constrained_layout=True,
                             figsize=(3.7 * len(tiers) + 1.0, 3.2 + 0.34 * k))
    axes = np.atleast_1d(axes)
    for ax, tier in zip(axes, tiers):
        r = oof(task, tier)
        cnt = confusion_matrix(df[f'y_{task}'].values, r['pred'], labels=labs)
        rate = cnt / cnt.sum(axis=1, keepdims=True)
        im = ax.imshow(rate, cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(k)); ax.set_yticks(range(k))
        ax.set_xticklabels(labs, rotation=40, ha='right'); ax.set_yticklabels(labs)
        for i in range(k):
            for j in range(k):
                if rate[i, j] >= 0.005:
                    ax.text(j, i, f'{rate[i, j]:.2f}\n{cnt[i, j]}', ha='center',
                            va='center', fontsize=7.5 if k <= 3 else 6.2,
                            color='white' if rate[i, j] > 0.5 else 'black')
        ax.set_title(f'{tier}   acc {T6.loc[(task, tier), "accuracy"]:.3f} | '
                     f'macro-F1 {T6.loc[(task, tier), "f1_macro"]:.3f}')
        ax.set_xlabel('predicted')
    axes[0].set_ylabel('true')
    fig.colorbar(im, ax=axes, shrink=.8, label='row-normalised rate')
    fig.suptitle(title or f'{task}: confusion by masking tier (out-of-fold)')
    save(fig, name)

# the paper's figure, then the complete view
plot_confusion('subtype', ['M0', 'M2'], 'V2_confusion_M0_vs_M2',
               'Subtype confusion, unmasked vs masked')
for task in TASKS:
    plot_confusion(task, TIERS, f'V9_confusion_{task}')

In [ ]:
for task in TASKS:
    labs = oof(task, 'M0')['classes']
    sub = T7[T7.task.eq(task)]
    fig, axes = plt.subplots(1, 3, constrained_layout=True, sharey=True,
                             figsize=(11, 2.8 + 0.34 * len(labs)))
    for ax, met in zip(axes, ['precision', 'recall', 'f1']):
        M = np.array([[sub[sub.tier.eq(t) & sub['class'].eq(c)][met].iloc[0]
                       for t in TIERS] for c in labs])
        im = ax.imshow(M, cmap='viridis', vmin=0, vmax=1, aspect='auto')
        ax.set_xticks(range(len(TIERS))); ax.set_xticklabels(TIERS)
        ax.set_yticks(range(len(labs))); ax.set_yticklabels(labs)
        for i in range(len(labs)):
            for j in range(len(TIERS)):
                ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                        fontsize=8, color='white' if M[i, j] < .65 else 'black')
        ax.set_title(met)
    fig.colorbar(im, ax=axes, shrink=.8)
    fig.suptitle(f'{task}: per-class metrics by masking tier ({REF_MODEL})')
    save(fig, f'V10_metric_heatmap_{task}')

## 16. What the classifier keys on (F3, V5)

Top positive coefficients per class, before and after masking. If M0 is
dominated by disclosure vocabulary and M2 by symptom or context language, that
is the qualitative evidence behind the quantitative drop.

These two fits are the only ones in the notebook that are not cross-validated,
because a coefficient vector is an object to inspect rather than a score to
estimate. The block prefix (`w__` word, `c__` char, `num__` dense) is kept in
the printed names: the earlier version stripped it, so a dense feature such as
`num__phq_mood` was displayed as if it were the word "mood".

In [ ]:
y_sub = df['y_subtype'].values
COEF = {}
for tier in ['M0', 'M2', 'M3']:
    pipe = make_pipe(MODELS[REF_MODEL]['factory'](None), False, 'v1')
    pipe.fit(FEATS[tier], y_sub)
    COEF[tier] = (np.array(pipe.named_steps['feat'].get_feature_names_out()),
                  pipe.named_steps['clf'].coef_,
                  list(pipe.named_steps['clf'].classes_))
    print(f'{tier}: fitted for interpretation')

out = {}
for tier, (names, coef, classes) in COEF.items():
    for ci, cls in enumerate(classes):
        out[f'{tier}__{cls}'] = names[np.argsort(coef[ci])[::-1][:12]]
F3 = pd.DataFrame(out)
F3.to_csv(f'{RES}/F3_top_features.csv', index=False)
print()
print(F3.to_string())

In [ ]:
FOCUS_CLASS = 'major depressive'   # try: bipolar, psychotic, postpartum, atypical
TOP_N = 14

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), sharex=True)
for ax, tier in zip(axes, ['M0', 'M2', 'M3']):
    names, coef, classes = COEF[tier]
    c = coef[classes.index(FOCUS_CLASS)]
    idx = np.argsort(c)[::-1][:TOP_N][::-1]
    ax.barh(range(len(idx)), c[idx], color=TIER_C[tier])
    ax.set_yticks(range(len(idx))); ax.set_yticklabels(names[idx], fontsize=8)
    ax.set_title(tier); ax.set_xlabel('logistic regression coefficient')
    ax.grid(axis='x', alpha=.3)
fig.suptitle(f'Top predictive features for "{FOCUS_CLASS}"', y=1.02)
fig.tight_layout()
save(fig, f'V5_top_features_{FOCUS_CLASS.replace(" ", "_")}')

## 17. Does leakage predict fragility? (V3)

AlSaad et al. write that near-ceiling postpartum and psychotic performance
*"likely reflects the presence of lexically salient indicators"*. That is a
testable claim: if true, the classes with the most retrieval-vocabulary
leakage should lose the most F1 when that vocabulary is masked.

Each point is one subtype. The control class has no own-label term and is
excluded, which leaves **five points**, so the correlation is reported with
its n attached. The earlier version printed rho and p with no such warning,
which invites more confidence than five points can carry.

In [ ]:
from scipy.stats import spearmanr, pearsonr

f1_m0 = np.array(H['M0'].values)
f1_m2 = np.array(H['M2'].values)
leak_map = dict(zip(T1['class'], T1['pct_own_term']))
pts = [(c, leak_map.get(c, np.nan), a - b, a, b)
       for c, a, b in zip(H.index, f1_m0, f1_m2)]
pts = [p for p in pts if not np.isnan(p[1])]
xs = np.array([p[1] for p in pts]); ys = np.array([p[2] for p in pts])

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10, 4))
axA.scatter(xs, ys, s=110, color=CB['red'], zorder=3, edgecolor='white', linewidth=1.2)
for c, x, d, a, b in pts:
    axA.annotate(c, (x, d), textcoords='offset points', xytext=(7, 5), fontsize=8)
if len(xs) > 2:
    m, c0 = np.polyfit(xs, ys, 1)
    xr = np.linspace(xs.min() - 3, xs.max() + 3, 50)
    axA.plot(xr, m * xr + c0, ls='--', color=CB['grey'], zorder=1)
    rho, pv = spearmanr(xs, ys)
    r, _ = pearsonr(xs, ys)
    axA.set_title(f'Spearman rho={rho:.2f} (p={pv:.3f}), Pearson r={r:.2f}, n={len(xs)}')
axA.set_xlabel('% of class containing its own label term')
axA.set_ylabel('F1 drop, M0 -> M2')
axA.grid(alpha=.3); axA.axhline(0, color='k', lw=.8)

order = np.argsort([p[2] for p in pts])[::-1]
yy = np.arange(len(order)); h = 0.38
axB.barh(yy + h/2, [pts[i][3] for i in order], h, label='M0', color=TIER_C['M0'])
axB.barh(yy - h/2, [pts[i][4] for i in order], h, label='M2', color=TIER_C['M2'])
axB.set_yticks(yy); axB.set_yticklabels([pts[i][0] for i in order])
axB.set_xlabel('per-class F1'); axB.set_title('Per-subtype F1 before and after masking')
axB.legend(); axB.grid(axis='x', alpha=.3)
fig.tight_layout()
save(fig, 'V3_leakage_vs_fragility')

print('\nper-subtype detail')
print(pd.DataFrame([{'class': c, 'pct_own_term': x, 'F1_M0': a, 'F1_M2': b, 'drop': d}
                    for c, x, d, a, b in pts]).round(4).to_string(index=False))

## 18. Is binary screening riding on length or format? (T9)

Binary screening keeps most of its performance once the vocabulary is masked,
which reads as genuine signal. Two collection artifacts could produce the same
result: control tweets are shorter (section 2), and `amp`, the residue of the
HTML entity `&amp;`, is a top control feature (section 16). Three checks:

- **Length alone.** A Logistic Regression that sees only the character and
  word counts, on the same folds. If it scores well, length is a usable
  shortcut on its own.
- **Length-matched re-scoring.** The stored out-of-fold predictions, scored
  again on a subsample in which every word-count bin holds the two classes in
  their overall ratio, so length carries no information about the label.
  Nothing is refitted.
- **The `amp` artifact.** How often each class contains the bare word `amp`,
  most likely what the dataset's punctuation stripping left of `&amp;`.

Topic is not tested: the control class's top features are sports and party
words (section 16), and no check on this one dataset can separate that from
depression language.

In [ ]:
from sklearn.model_selection import cross_val_predict

def length_matched(n_words, y, rng, n_bins=10):
    """Indices of a subsample in which every word-count bin holds the classes
    in their overall ratio, so length carries no information about the label."""
    bins = pd.qcut(n_words, n_bins, labels=False, duplicates='drop')
    classes, counts = np.unique(y, return_counts=True)
    share = counts / counts.sum()
    keep = []
    for b in np.unique(bins):
        idx = [np.flatnonzero((bins == b) & (y == c)) for c in classes]
        # the largest bin size that keeps every class at its overall share
        size = min(len(i) / s for i, s in zip(idx, share))
        keep += [rng.choice(i, int(round(size * s)), replace=False)
                 for i, s in zip(idx, share)]
    return np.sort(np.concatenate(keep))

print('confound helpers ready')

In [ ]:
rows = []

# (a) length alone, on the same folds as every grid cell
length_pipe = Pipeline([
    ('feat', ColumnTransformer([('num', StandardScaler(), ['n_chars', 'n_words'])])),
    ('clf', LogisticRegression(max_iter=3000))])
for task in TASKS:
    y = df[f'y_{task}'].values
    pred = cross_val_predict(length_pipe, FEATS['M0'], y,
                             cv=StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                                random_state=SEED))
    rows.append({'check': 'length only', 'metric': 'macro_f1', 'task': task,
                 'tier': 'M0', 'model': 'LogisticRegression v1', 'n': len(y),
                 'value': f1_score(y, pred, average='macro')})

# (b) the stored binary predictions, re-scored on a length-matched subsample
y_bin = df['y_binary'].values
n_words = FEATS['M0']['n_words'].values
keep = length_matched(n_words, y_bin, np.random.default_rng(SEED))
for name, idx in [('all tweets', np.arange(len(y_bin))), ('length-matched', keep)]:
    means = {c: round(float(n_words[idx][y_bin[idx] == c].mean()), 1)
             for c in np.unique(y_bin)}
    print(f'{name:<15} n={len(idx):5d}  mean words by class {means}')
for tier in TIERS:
    for model, features, weight in [(REF_MODEL, 'v1', 'none'),
                                    ('HistGradBoost', 'v2', 'balanced')]:
        pred = oof('binary', tier, model, features, weight)['pred']
        for name, idx in [('all tweets', np.arange(len(y_bin))), ('length-matched', keep)]:
            rows.append({'check': name, 'metric': 'macro_f1', 'task': 'binary',
                         'tier': tier, 'model': f'{model} {features}', 'n': len(idx),
                         'value': f1_score(y_bin[idx], pred[idx], average='macro')})

# (c) the bare word 'amp', by class ('&amp;' itself never survives in the release)
amp = df['Tweets'].str.contains(r'\bamp\b', case=False, regex=True)
for lab, rate in amp.groupby(df['Labels']).mean().items():
    rows.append({'check': "word 'amp' in raw text", 'metric': 'pct_of_class',
                 'task': 'subtype', 'tier': 'M0', 'model': lab,
                 'n': int(df['Labels'].eq(lab).sum()), 'value': 100 * rate})

T9 = pd.DataFrame(rows).round(4)
T9.to_csv(f'{RES}/T9_binary_confound.csv', index=False)
print()
print(T9.to_string(index=False))

## 19. Robustness: tuned C and repeated folds (T10)

Every model above runs at a fixed C. A fair objection is that the masked
models are simply under-tuned. For the two linear models this section picks C
by nested cross-validation (an inner 3-fold search over `TUNE_C` on each
outer-training split) at M0, M2 and M3. It also reshuffles the 5-fold split
`REPEATS - 1` more times, so each M0-to-masked comparison rests on 15 paired
folds instead of 5.

These cells run outside the main grid on purpose, so tables T2-T8 are
unchanged by them.

In [ ]:
TUNE_TIERS = ['M0', 'M2', 'M3']
LINEAR = ['LogisticRegression', 'LinearSVM']

def x_cell(task, tier, model, repeat=0):
    c = dict(task=task, train_tier=tier, test_tier=tier, features='v1',
             weight='none', model=model)
    return dict(c, repeat=repeat) if repeat else c

X_GRID = [x_cell(task, tier, m + suffix, rep)
          for task in TASKS for tier in TUNE_TIERS for m in LINEAR
          for suffix, rep in [('_tuned', 0)] + [('', r) for r in range(1, REPEATS)]]

OOF_X = {}
t_start = time.time()
for i, c in enumerate(X_GRID, 1):
    r = run_cell(c)
    OOF_X[cfg_key(c)] = r
    tag = 'cached' if r['cached'] else f'{r["secs"]:.0f}s'
    print(f'[{i:3d}/{len(X_GRID)}] {cfg_key(c):<74s} {tag}')
print(f'\nrobustness grid complete in {(time.time()-t_start)/60:.1f} min')

In [ ]:
rows = []
for task in TASKS:
    y = df[f'y_{task}'].values
    for m in LINEAR:
        for tier in TUNE_TIERS:
            base = R[R.task.eq(task) & R.model.eq(m) & R.tier.eq(tier)].iloc[0]
            tuned = OOF_X[cfg_key(x_cell(task, tier, m + '_tuned'))]
            folds = list(base['folds'])
            for rep in range(1, REPEATS):
                r = OOF_X[cfg_key(x_cell(task, tier, m, rep))]
                folds += per_fold_macro_f1(y, r['pred'], r['fold'])
            rows.append({'task': task, 'model': m, 'tier': tier,
                         'default_C': base['macro_f1'],
                         'tuned_C': float(np.mean(per_fold_macro_f1(y, tuned['pred'],
                                                                    tuned['fold']))),
                         'chosen_C': ' '.join(f'{v:g}' for v in tuned['best_c']),
                         'repeated_mean': float(np.mean(folds)),
                         'repeated_std': float(np.std(folds, ddof=1)),
                         'folds': folds})
T10 = pd.DataFrame(rows)

drops = []
for (task, m), g in T10.groupby(['task', 'model']):
    g = g.set_index('tier')
    for tier in TUNE_TIERS[1:]:
        t, p = ttest_rel(g.loc['M0', 'folds'], g.loc[tier, 'folds'])
        drops.append({'task': task, 'model': m, 'tier': tier,
                      'drop_default': g.loc['M0', 'default_C'] - g.loc[tier, 'default_C'],
                      'drop_tuned': g.loc['M0', 'tuned_C'] - g.loc[tier, 'tuned_C'],
                      'drop_repeated': g.loc['M0', 'repeated_mean'] - g.loc[tier, 'repeated_mean'],
                      't_repeated': float(t), 'p_repeated': float(p),
                      'n_pairs': len(g.loc[tier, 'folds'])})
T10d = pd.DataFrame(drops).round(4)

T10.drop(columns='folds').round(4).to_csv(f'{RES}/T10_tuning_repeats.csv', index=False)
T10d.to_csv(f'{RES}/T10_robust_drops.csv', index=False)
print(T10.drop(columns='folds').round(4).to_string(index=False))
print()
print(T10d.to_string(index=False))
print('\nIf drop_tuned matches drop_default, the masking cost is not an artifact')
print('of a fixed C; p_repeated tests the drop over all repeated fold pairs.')

## 20. Export summary

In [ ]:
print('=== HEADLINE NUMBERS ===')
for task in TASKS:
    t = T2.loc[task]
    b = B[B.task.eq(task)]
    print(f'\n[{task}]  n={len(df)}')
    print(f"  majority floor     : {b[b.baseline.eq('majority')]['macro_f1'].iloc[0]:.4f}")
    print(f"  keyword-rule floor : {b[b.baseline.eq('keyword_rule')]['macro_f1'].iloc[0]:.4f}")
    print(f'  best M0            : {t["M0"].max():.4f}  ({t["M0"].idxmax()})')
    print(f'  best M2            : {t["M2"].max():.4f}  ({t["M2"].idxmax()})')
    print(f'  best M2 + v2       : {T4.loc[(task, "M2"), "v2_best"]:.4f}'
          f'  ({T4.loc[(task, "M2"), "v2_best_model"]})')
    print(f'  best M3            : {t["M3"].max():.4f}  ({t["M3"].idxmax()})')
    print(f'  best M3 + v2       : {T4.loc[(task, "M3"), "v2_best"]:.4f}'
          f'  ({T4.loc[(task, "M3"), "v2_best_model"]})')
    print(f'  largest drop       : {t["delta_M0_M2"].max():.4f}  ({t["delta_M0_M2"].idxmax()})')
    print(f'  leaky-pipeline lift: '
          f'{T3[T3.task.eq(task)]["inflation"].iloc[0]:+.4f}')

print('\n=== TABLES ===')
for f in sorted(os.listdir(RES)):
    if f.endswith('.csv'):
        print('   results/' + f)
print('\n=== FIGURES ===')
for f in sorted(os.listdir(FIGS)):
    if f.endswith('.pdf'):
        print('   results/figures/' + f)
print(f'\ntotal grid compute: {ALL["secs"].sum()/60:.1f} min')